# Irodori-TTS-Server (Google Colab)
絵文字でしゃべり方を制御できる日本語TTSモデル**Irodori-TTS**をColabで動かすノートブックです。

モデルをGoogle Driveに保存するので2回目以降は少し早く起動できます。

使い方:ステップ1から順にセルを実行してください(▶ボタン or Shift+Enter)

⚠️ メニューの `ランタイム → ランタイムのタイプを変更` から**T4 GPU**を選択してください。

## ステップ1: Google Driveをマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## ステップ2: 環境変数の設定
参照元になる音声ファイルは事前に準備しておいてください。

In [ ]:
import os

# ベースフォルダ
DRIVE_BASE_DIR = "/content/drive/MyDrive/Irodori-TTS"

# 各種保存先フォルダ
os.environ["HF_HOME"] = os.path.join(DRIVE_BASE_DIR, "huggingface_cache")
DRIVE_VOICES_DIR = os.path.join(DRIVE_BASE_DIR, "voices")

# ローカルパス
REPO_DIR = "/content/Irodori-TTS-Server"
VENV_DIR = "/content/.venv"

print("✅ パス変数の定義完了")

# ベースフォルダや音声フォルダがなければ自動作成
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
os.makedirs(DRIVE_VOICES_DIR, exist_ok=True)

print(f"⚠️ モデルの保存先はGoogle Driveの '{os.environ['HF_HOME']}' になります")
print(f"⚠️ 参照したい音声ファイル(.wav)を '{DRIVE_VOICES_DIR}' にアップロードして配置してください")

## ステップ3: リポジトリ取得


In [ ]:
%cd /content

!rm -rf Irodori-TTS-Server
!git clone https://github.com/Aratako/Irodori-TTS-Server.git

## ステップ4: uvのインストール

In [ ]:
!pip -q install -U uv

## ステップ5: 仮想環境と依存ライブラリ
この工程は完了するまで少し時間がかかります。

In [ ]:
%cd /content/Irodori-TTS-Server

!uv sync --extra cu128

## ステップ6: voicesフォルダを紐付け
SillyTavernの**Available Voices**にはボイス一覧に表示された名前(ボイス無しの場合は**none**)を入力します。

Google Driveにwavファイルがアップロードされていなければ無効になります。

In [ ]:
import os

SERVER_VOICES_DIR = os.path.join(REPO_DIR, "voices")

if os.path.exists(SERVER_VOICES_DIR):
    if os.path.islink(SERVER_VOICES_DIR):
        os.unlink(SERVER_VOICES_DIR)
    else:
        !rm -rf {SERVER_VOICES_DIR}

os.symlink(DRIVE_VOICES_DIR, SERVER_VOICES_DIR)
print("✅ シンボリックリンク完了")

available_voices = [
    os.path.splitext(f)[0]
    for f in os.listdir(DRIVE_VOICES_DIR)
    if f.endswith(('.wav'))
]

if available_voices:
    voices_comma_separated = ",".join(available_voices)

    print("\n📋 【現在利用可能なボイス一覧】")
    print(voices_comma_separated)
else:
    print("⚠️ 音声ファイルはありません")
    print("none")

## ステップ7: cloudflaredを取得

In [ ]:
%cd /content

!wget -q -O cloudflared \
    https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64

!chmod +x cloudflared

!./cloudflared --version

## ステップ8: Irodori-TTS-Server起動

In [ ]:
import subprocess
import os
import time

SERVER_LOG = "/content/server.log"

# 古いサーバープロセスがあれば強制終了
!pkill -f irodori_openai_tts || true
time.sleep(1)

if os.path.exists(SERVER_LOG):
    os.remove(SERVER_LOG)

print("⏳ Irodori-TTS-Serverをバックグラウンドで起動...")

env = os.environ.copy()
env["IRODORI_CORS_ORIGINS"] = '["*"]'

with open(SERVER_LOG, "w") as f:
    server = subprocess.Popen(
        ["uv", "run", "python", "-m", "irodori_openai_tts",
         "--host", "0.0.0.0", "--port", "8088"],
        cwd=REPO_DIR,
        env=env,
        stdout=f,
        stderr=subprocess.STDOUT,
        text=True,
    )

print("✅ サーバープロセスを開始(ログは '/content/server.log' に随時記録されます)")

## ステップ9: 起動確認とウォームアップ
一度も使ったことがない場合はモデルのダウンロードでさらに時間がかかります。

In [ ]:
import time
import requests
import os

print("⏳ 起動確認をしています...")
timeout = 120
start_time = time.time()
is_ready = False

while time.time() - start_time < timeout:
    try:
        response = requests.get("http://127.0.0.1:8088/health")
        if response.status_code == 200:
            print("✅ サーバーが正常に起動")
            is_ready = True
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(3)

if is_ready:
    print("\n⏳ ウォームアップを実行中...")

    test_voice = "none"
    if 'available_voices' in locals() and available_voices:
        test_voice = os.path.splitext(available_voices[0])[0]

    headers = {"Content-Type": "application/json"}
    payload = {
        "model": "irodori-tts",
        "input": "あ",
        "voice": test_voice
    }

    warmup_start = time.time()
    try:
        res = requests.post("http://127.0.0.1:8088/v1/audio/speech", json=payload, headers=headers, timeout=300)

        if res.status_code == 200:
            print(f"✅ 【成功】モデルのロード･ダウンロードが全て完了しました(所要時間: {int(time.time() - warmup_start)}秒)")

            from IPython.display import display, Audio
            display(Audio(res.content, autoplay=False))
        else:
            print(f"⚠️ 【リクエストエラー】(Status: {res.status_code})")
            print(f"レスポンス: {res.text} ※裏ではダウンロードが継続している可能性があります")

    except requests.exceptions.Timeout:
        print("❌ 【タイムアウト】モデルのダウンロードに5分以上かかっているかフリーズしている可能性があります\n")
        !tail -n 20 /content/server.log
    except Exception as e:
        print(f"⚠️ リクエスト中に予期せぬエラーが発生しました: {e}\n")
        !tail -n 20 /content/server.log
else:
    print("❌ サーバー自体の起動に失敗した可能性があります\n")
    !tail -n 15 /content/server.log

## ステップ10: Cloudflare Tunnel公開

In [ ]:
import subprocess
import re
import time
from IPython.display import display, HTML

# 古いcloudflaredプロセスを強制終了
!pkill -f cloudflared
time.sleep(1)

print("✅ Cloudflare Tunnelを起動して外部公開用URLを生成")

cloudflared = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8088"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

tunnel_url = None

# Cloudflareの出力を監視してURLを抽出する
for line in cloudflared.stdout:

    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        endpoint_url = f"{tunnel_url}/v1/audio/speech"

        # URLコピー用のUIを表示
        html_ui = f"""
        <div style="margin: 20px 0; padding: 18px; border: 1px solid #4285F4; border-radius: 8px; background-color: #f8fafd; font-family: sans-serif; max-width: 650px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
            <h3 style="margin-top: 0; color: #1a73e8; font-size: 16px; font-weight: bold;">接続用URLが生成されました</h3>
            <p style="margin: 5px 0 12px 0; font-size: 13px; color: #5f6368;">
                右側の「コピー」ボタンをクリックするとエンドポイントURLがクリップボードに保存されます
            </p>

            <div style="display: flex; align-items: center; background: #ffffff; padding: 10px; border: 1px solid #dadce0; border-radius: 6px; box-shadow: inset 0 1px 2px rgba(0,0,0,0.05); justify-content: space-between;">
                <code id="target-url" style="font-family: monospace; font-size: 13px; color: #202124; word-break: break-all; margin-right: 15px; font-weight: bold;">{endpoint_url}</code>
                <button onclick="copyUrl()" style="padding: 8px 16px; background-color: #1a73e8; color: white; border: none; border-radius: 4px; cursor: pointer; font-weight: bold; font-size: 13px; white-space: nowrap; transition: background-color 0.2s;">
                    コピー
                </button>
            </div>

            <div id="copy-message" style="margin-top: 8px; font-size: 13px; color: #188038; font-weight: bold; height: 18px;"></div>

            <p style="margin: 0; font-size: 13px; color: #3c4043; line-height: 1.5;">
                <b>※失敗した場合は手動で選択してコピーしてください</b>
            </p>
        </div>

        <script>
        function copyUrl() {{
            const text = document.getElementById("target-url").innerText;
            navigator.clipboard.writeText(text).then(() => {{
                const msg = document.getElementById("copy-message");
                msg.innerText = "📋 クリップボードにコピーしました";
                setTimeout(() => {{ msg.innerText = ""; }}, 4000);
            }}).catch(err => {{
                alert("コピーに失敗しました");
            }});
        }}
        </script>
        """
        display(HTML(html_ui))
        print("🌐 トンネルを維持しています...")
        break

# URL抽出後はエラー発生時のみログを表示するように待機
try:
    for line in cloudflared.stdout:
        if "error" in line.lower():
            print(f"[Tunnel Log] {line.strip()}")
except KeyboardInterrupt:
    print("🛑 トンネルを停止します")
    cloudflared.terminate()

return_code = cloudflared.poll()
if return_code is not None and return_code != 0:
    print(f"❌ Cloudflareがコード {return_code} で強制終了しました")